In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df

lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df

supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Define time boundaries once
ts = pd.Timestamp("1996-01-01")
end = pd.Timestamp("1996-04-01")  # 3 months after 1996-01-01

# Filter and compute revenue parts in one chained, vectorized pass
lineitem_filtered = (
    lineitem.loc[(lineitem.L_SHIPDATE >= ts) & (lineitem.L_SHIPDATE < end),
                  ["L_SUPPKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]]
    .assign(REVENUE_PARTS=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
    .loc[:, ["L_SUPPKEY", "REVENUE_PARTS"]]
)

# Aggregate total revenue per supplier
revenue_table = (
    lineitem_filtered
    .groupby("L_SUPPKEY", as_index=False, sort=False)
    .agg(TOTAL_REVENUE=("REVENUE_PARTS", "sum"))
    .rename(columns={"L_SUPPKEY": "SUPPLIER_NO"})
)

# Select only the supplier(s) with the maximum revenue
max_revenue = revenue_table["TOTAL_REVENUE"].max()
revenue_table = revenue_table.loc[revenue_table.TOTAL_REVENUE == max_revenue]

# Filter supplier columns and merge
supplier_filtered = supplier[["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE"]]

total = (
    supplier_filtered
    .merge(revenue_table, left_on="S_SUPPKEY", right_on="SUPPLIER_NO", how="inner")
    [["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE", "TOTAL_REVENUE"]]
)